# BISINDO Tahap 1b — exploratory/development
Persetujuan peneliti: **2026-11-03**, eksekusi oleh peneliti, bukan agen.

Target: **10 full runs A/B ×seed42–46, 224px/30epoch +10 linear head fits +masking eval +kurva**.
Semua seed dilaporkan; seed42 **replikasi**. `last.pt` fixed. Tidak ada Kaggle, tuning, overwrite, resume, atau Tahap2.

Gunakan kernel `.venv-bisindo/Scripts/python.exe`, cwd=root proyek. Modul `.py`, cache v3, pretrained pinned, dan data internal wajib tersedia.
Notebook ini baru; notebook dan run sebelumnya tidak diubah.

**Urutan:** config → definisi → fase pertama seed42 → baca waktu → ubah flag lanjut → seed43–46 → diagnosis → laporan.
`RUN_TRAINING=False` default. Jangan memakai Run All untuk melewati pemeriksaan waktu pertama.
Jika gagal: baca tabel/error, simpan artefak, hentikan; jangan resume atau mengganti seed. Run parsial tetap dilaporkan.

In [1]:
from pathlib import Path
from dataclasses import asdict
import json
import sys
import time
import pandas as pd
from IPython.display import display, Markdown, Image
import classification_research as research
import classification_diagnostics as diagnostics
import classification_workflow as workflow

ROOT = Path.cwd().resolve()
TAG = 'exploratory_development_stage1b_user_20261103_v1'
RUN_TRAINING = True  # Ubah True hanya saat Anda siap menjalankan Tahap1b yang telah disetujui.
RUN_DIAGNOSTICS = False
APPROVED_BY = 'researcher: explicit approval in chat 2026-11-03'
WORKS = workflow.seed_works(ROOT, TAG)
SESSION = ROOT / (TAG + '_session')
assert Path(sys.executable).resolve() == (ROOT / '.venv-bisindo/Scripts/python.exe').resolve(), 'Pilih kernel .venv-bisindo'
assert research.base.b.sha(ROOT/'PREREGISTRATION.md') == (ROOT/'PREREGISTRATION.sha256').read_text().split()[0], 'Protocol hash drift'
CONTINUE_AFTER_TIMING = False
print('exploratory/development | RUN_TRAINING =', RUN_TRAINING)


exploratory/development | RUN_TRAINING = True


## Orkestrasi — logika eksperimen tetap dalam modul
Approval files baru dibuat **hanya ketika RUN_TRAINING=True**, berdasarkan persetujuan eksplisit di atas,
terikat work/settings/hash kandidat. Notebook tidak mengubah protokol/addendum.

Ledger mempunyai20 baris sejak awal. Snapshot CSV bernomor bersifat immutable; JSONL append-only menyimpan status/error/timing.
Kegagalan teknis menghentikan eksekusi, termasuk interruption. Sel laporan tetap dapat dijalankan untuk melaporkan hasil parsial.
Full-run wall time dicatat pada level **pasangan +evaluasi**, bukan dialokasikan palsu ke A/B.
Waktu training masing-masing A/B berasal dari `results.csv`; waktu probe berasal dari pengukuran call aktual.

In [2]:
def snapshot(event):
    event = dict(status_label='exploratory/development', **event)
    with (SESSION/'exploratory_events.jsonl').open('a', encoding='utf8') as stream:
        stream.write(json.dumps(event, default=str) + '\n')
    revision = len(list(SESSION.glob('exploratory_fit_table_*.csv')))
    path = SESSION / f'exploratory_fit_table_{revision:03d}.csv'
    with path.open('x', encoding='utf8', newline='') as stream:
        pd.DataFrame(ledger).to_csv(stream, index=False)


def initialize_session():
    global ledger, approvals, phase_times
    for work in WORKS.values():
        research.new_work(ROOT, work)
    for seed in workflow.SEEDS:
        for model in ['A','B']:
            assert not (ROOT/f'{TAG}_probe_{model}_seed{seed}').exists(), 'Probe output exists'
    SESSION.mkdir(exist_ok=False)
    ledger = [dict(kind=kind, model=model, seed=seed, replication=(seed==42),
                   status='not_started', seconds=None, common6_macro_f1=None,
                   common6_accuracy=None, full8_macro_f1=None, full8_accuracy=None,
                   error='', status_label='exploratory/development')
              for seed in workflow.SEEDS for kind in ['full','linear_probe'] for model in ['A','B']]
    approvals = {}; phase_times = []
    protocol_hash = research.base.b.sha(ROOT/'PREREGISTRATION.md')
    for seed, work in WORKS.items():
        record = dict(kind='bisindo_training_approval', scope='stage1b', approved_by=APPROVED_BY,
                      work_names=[work.name], training=asdict(research.Settings(seed,30,224)),
                      protocol_sha256={'PREREGISTRATION.md':protocol_hash}, linear_probes=True,
                      status='exploratory/development', approval_date='2026-11-03',
                      approval_source='explicit researcher chat; researcher executes this notebook')
        approvals[seed] = SESSION/f'exploratory_approval_seed{seed}.json'
        with approvals[seed].open('x', encoding='utf8') as stream:
            json.dump(record, stream, indent=2)
    snapshot(dict(event='session_initialized', training_authorized=True, seed42_is_replication=True))


def entry(seed, kind, model):
    return next(row for row in ledger if (row['seed'],row['kind'],row['model'])==(seed,kind,model))


def run_seed(seed):
    if not RUN_TRAINING:
        raise PermissionError('RUN_TRAINING=False')
    work = WORKS[seed]
    if any(row['status']!='not_started' for row in ledger if row['seed']==seed):
        raise RuntimeError('Seed already attempted; no resume/overwrite')
    research.new_work(ROOT, work)
    started = time.perf_counter(); current_kind='full'; current_model=None
    phase = dict(seed=seed, replication=(seed==42), status='running')
    try:
        tick=time.perf_counter()
        ready=research.prepare(ROOT,ROOT/'experiment_yolo26_v3',work,research.Settings(seed,30,224))
        phase['prepare_seconds']=time.perf_counter()-tick
        if not ready['ready']:raise RuntimeError('Technical readiness blocked: '+str(ready))
        for model in ['A','B']:entry(seed,'full',model)['status']='running'
        snapshot(dict(event='pair_started',seed=seed))
        tick=time.perf_counter()
        research.train(ROOT,ROOT/'experiment_yolo26_v3',work,approval=approvals[seed])
        phase['pair_train_and_evaluation_seconds']=time.perf_counter()-tick
        metrics=pd.read_csv(work/'evaluation/comparison_metrics.csv')
        for model in ['A','B']:
            row=entry(seed,'full',model)
            curve=pd.read_csv(work/'runs'/f'model_{model}'/'results.csv')
            curve.columns=curve.columns.str.strip()
            assert curve.epoch.tolist()==list(range(1,31)), 'Fixed30 epochs incomplete'
            old=metrics[(metrics.model==model)&(metrics.scope=='common6')].iloc[0]
            row.update(status='completed',seconds=float(curve.time.iloc[-1]),
                       common6_macro_f1=float(old.macro_f1),common6_accuracy=float(old.accuracy))
            if model=='B':
                full=metrics[(metrics.model==model)&(metrics.scope=='full8')].iloc[0]
                row.update(full8_macro_f1=float(full.macro_f1),full8_accuracy=float(full.accuracy))
        snapshot(dict(event='pair_completed',seed=seed,**{'timing':phase.copy()}))
        current_kind='linear_probe'
        for model in ['A','B']:
            current_model=model; row=entry(seed,current_kind,model);row['status']='running'
            snapshot(dict(event='probe_started',seed=seed,model=model))
            tick=time.perf_counter()
            result=diagnostics.fit_linear_probe(work,model,ROOT/f'{TAG}_probe_{model}_seed{seed}',approval=approvals[seed])
            duration=time.perf_counter()-tick
            row.update(status='completed',seconds=duration,common6_macro_f1=result['macro_f1'],common6_accuracy=result['accuracy'])
            phase[f'probe_{model}_seconds']=duration
            snapshot(dict(event='probe_completed',seed=seed,model=model,seconds=duration))
        phase['status']='completed'
    except BaseException as exc:
        message=f'{type(exc).__name__}: {exc}'
        phase.update(status='failed',error=message)
        if current_kind=='full':
            for model in ['A','B']:
                row=entry(seed,'full',model)
                if row['status']=='completed':continue
                folder=work/'runs'/f'model_{model}'
                csv=folder/'results.csv'
                complete=False
                if csv.is_file() and (folder/'weights/last.pt').is_file():
                    curve=pd.read_csv(csv);curve.columns=curve.columns.str.strip()
                    complete=curve.epoch.tolist()==list(range(1,31))
                    row['seconds']=float(curve.time.iloc[-1])
                row.update(status='completed_unreleased' if complete else ('failed' if folder.exists() else 'not_run_after_failure'),error=message)
        else:
            entry(seed,current_kind,current_model).update(status='failed',error=message)
        snapshot(dict(event='hard_stop',seed=seed,kind=current_kind,model=current_model,error=message))
        raise
    finally:
        phase['total_wall_seconds']=time.perf_counter()-started
        phase_times.append(phase)
        snapshot(dict(event='phase_timing',**phase))
        display(pd.DataFrame(phase_times))
        display(pd.DataFrame(ledger))
    return phase


## Fase pertama — seed42 replikasi, dua full runs +dua probes
Set `RUN_TRAINING=True` pada config, lalu jalankan config/definisi/sel ini.
Prepare hanya seed42 dulu; tidak membuka Kaggle. Setelah selesai, **baca tabel waktu yang ditampilkan dan berhenti di sini**.
Nilai belum diisi sebelumnya; semuanya diukur saat Anda menjalankan notebook.
Jika sel gagal, jalankan sel laporan parsial, lalu hentikan. Jangan jalankan sisa seed.

In [3]:
if RUN_TRAINING:
    initialize_session()
    first_phase = run_seed(42)
    CONTINUE_AFTER_TIMING = False
    display(Markdown('**PAUSE: waktu fase pertama tercatat di atas.** Baca hasil sebelum mengubah flag lanjut pada sel berikut.'))
else:
    print('SKIPPED: RUN_TRAINING=False; zero fits. Tidak ada work dir/approval yang dibuat.')


Classification automatic frames: 40/640
Classification automatic frames: 80/640
Classification automatic frames: 120/640
Classification automatic frames: 160/640
Classification automatic frames: 200/640
Classification automatic frames: 240/640
Classification automatic frames: 280/640
Classification automatic frames: 320/640
Classification automatic frames: 360/640
Classification automatic frames: 400/640
Classification automatic frames: 440/640
Classification automatic frames: 480/640
Classification automatic frames: 520/640
Classification automatic frames: 560/640
Classification automatic frames: 600/640
Classification automatic frames: 640/640
Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\exploratory_development_stage1b_user_20261103_v1_seed42\ultralytics_config\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. F

,seed,replication,status,prepare_seconds,pair_train_and_evaluation_seconds,error,total_wall_seconds
0,42,True,failed,71.912502,139.929736,RuntimeError: Ultralytics already imported wit...,215.366961


,kind,model,seed,replication,status,seconds,common6_macro_f1,common6_accuracy,full8_macro_f1,full8_accuracy,error,status_label
0,full,A,42,True,completed,53.1245,0.931776,0.933333,NaN,NaN,,exploratory/development
1,full,B,42,True,completed,67.1860,0.930042,0.925000,0.935464,0.9375,,exploratory/development
2,linear_probe,A,42,True,failed,NaN,NaN,NaN,NaN,NaN,RuntimeError: Ultralytics already imported wit...,exploratory/development
3,linear_probe,B,42,True,not_started,NaN,NaN,NaN,NaN,NaN,,exploratory/development
4,full,A,43,False,not_started,NaN,NaN,NaN,NaN,NaN,,exploratory/development
5,full,B,43,False,not_started,NaN,NaN,NaN,NaN,NaN,,exploratory/development
6,linear_probe,A,43,False,not_started,NaN,NaN,NaN,NaN,NaN,,exploratory/development
7,linear_probe,B,43,False,not_started,NaN,NaN,NaN,NaN,NaN,,exploratory/development
8,full,A,44,False,not_started,NaN,NaN,NaN,NaN,NaN,,exploratory/development
9,full,B,44,False,not_started,NaN,NaN,NaN,NaN,NaN,,exploratory/development


RuntimeError: Ultralytics already imported with different settings directory; run pilot CLI in a fresh process

## Lanjut seed43–46 — setelah meninjau waktu pertama
Ubah **hanya setelah membaca waktu fase pertama**: `CONTINUE_AFTER_TIMING=True` di sel berikut.
Ini melanjutkan seed **yang belum dimulai**, bukan resume fit. Restart kernel / kehilangan state tidak mendukung resume.
Semua5 seed tetap direncanakan; seed42 tidak dihapus atau diganti karena hasilnya.
Kegagalan apa pun menghentikan loop; ledger mempertahankan semua20 baris, termasuk yang belum dijalankan.

In [ ]:
CONTINUE_AFTER_TIMING = False  # Ubah True secara manual setelah membaca waktu seed42.
if RUN_TRAINING and CONTINUE_AFTER_TIMING:
    assert first_phase['status']=='completed', 'Fase pertama belum selesai'
    assert all(entry(42,kind,model)['status']=='completed' for kind in ['full','linear_probe'] for model in ['A','B'])
    for seed in [43,44,45,46]:
        run_seed(seed)
else:
    print('PAUSED: sisa seed belum dijalankan.')


## Masking eval, paired analysis dan kurva — tanpa fit tambahan
Sesudah20 fit lengkap, ubah `RUN_DIAGNOSTICS=True` pada config **tanpa menjalankan ulang config atau sel training**
(atau set nilai itu pada sel berikut). Modul memeriksa hash/GT/split dan memakai last.pt.
Union landmarks15%/gray114 sebelum letterbox; missing landmarks tetap denominator/strata tersendiri.
Chance A6=1/6, B8=1/8 sebagai uniform-output reference. Context-only masih dapat memuat wajah/tubuh.

Common6/full8 terpisah. Bootstrap10000/RNG20261103 memakai shared video draw lintas seed pairs.
Pooled McNemar seed-video hanya deskriptif: repeated video bukan sampel independen.
Kurva train actual weights versus validation EMA dilabeli; tidak mengganti last.pt dengan best.pt.

In [ ]:
RUN_DIAGNOSTICS = False  # Ubah True setelah semua20 fit completed.
if RUN_DIAGNOSTICS:
    assert 'ledger' in globals() and len(ledger)==20
    assert all(row['status']=='completed' for row in ledger), 'Fit gagal/tidak lengkap: hentikan; laporkan ledger'
    tick=time.perf_counter()
    try:
        multiseed=workflow.diagnose_stage1b(ROOT,TAG)
        snapshot(dict(event='diagnostics_completed',seconds=time.perf_counter()-tick))
        display(multiseed)
        masked=pd.concat([pd.read_csv(ROOT/f'{TAG}_masked_eval_seed{seed}'/'exploratory_masked_metrics.csv').assign(seed=seed)
                          for seed in workflow.SEEDS],ignore_index=True)
        display(masked)
        display(Image(filename=str(ROOT/f'{TAG}_curves_seed42'/'exploratory_learning_curves.png')))
    except BaseException as exc:
        snapshot(dict(event='diagnostics_failed',error=f'{type(exc).__name__}: {exc}',seconds=time.perf_counter()-tick))
        raise
else:
    print('Masking eval/paired/kurva belum dijalankan. Tidak ada Tahap2.')


## Laporan akhir atau parsial — FACT / HYPOTHESIS / PROPOSAL
Jalankan juga setelah kegagalan. Jangan mengisi angka hilang dengan tebakan.
Laporan final hanya diberi status complete bila20 fit **dan** diagnosis lengkap.
Tabel merangkum setiap seed/failure; statistik mean±SD hanya dari fit selesai, dengan jumlah fit dilaporkan.
Jika ada fit gagal/tidak lengkap, agregat tidak menjadi hasil lima-seed lengkap.
Tidak ada Tahap2 otomatis, apa pun hasilnya.

In [ ]:
if 'ledger' in globals():
    table=pd.DataFrame(ledger)
    display(table)
    events=[json.loads(line) for line in (SESSION/'exploratory_events.jsonl').read_text(encoding='utf8').splitlines()]
    errors=[event for event in events if event.get('event') in ['hard_stop','diagnostics_failed']]
    completed=table[table.status=='completed'].copy()
    scores=completed.melt(id_vars=['kind','model','seed'],value_vars=['common6_macro_f1','full8_macro_f1'],
                          var_name='scope_metric',value_name='macro_f1').dropna()
    aggregate=scores.groupby(['kind','model','scope_metric']).macro_f1.agg(['count','mean','std']).reset_index()
    display(aggregate)
    summary_path=ROOT/f'{TAG}_exploratory_multiseed'/'exploratory_multiseed_summary.json'
    summary=json.loads(summary_path.read_text(encoding='utf8')) if summary_path.is_file() else None
    complete=(len(table)==20 and table.status.eq('completed').all() and summary is not None and not errors)
    facts=dict(status='complete' if complete else 'partial_or_not_diagnosed',label='exploratory/development',
               all20fits=table.status.eq('completed').all().item(),diagnostics_complete=summary is not None,
               phase_times=phase_times,failures=errors,multiseed=summary,proceed_to_stage2=False)
    report=SESSION/'exploratory_final_report.md'
    text=('# exploratory/development — Tahap1b '+facts['status']+'\n\n## FACT\n\n'
          +workflow.paired.markdown_table(table)+'\n\n'
          +workflow.paired.markdown_table(aggregate)+'\n\n```json\n'
          +json.dumps(facts,indent=2,default=str)+'\n```\n\n'
          +'## HYPOTHESIS\n\nContext cues/weak labels/overfit adalah kemungkinan penjelasan, bukan bukti sebab. '
          +'Baca masking, probe dan kurva bersama keterbatasan signer/session dan distribution shift.\n\n'
          +'## PROPOSAL\n\nHentikan setelah Tahap1b. Tidak memilih seed/checkpoint atau tuning Kaggle. '
          +'Jika SD seed >=abs(mean paired delta), atau CI melintasi -delta kandidat, pilot tidak memutuskan. '
          +'Tidak mengubah delta/H2 dari hasil ini. Tahap2 memerlukan keputusan peneliti dan addendum terlebih dahulu.\n')
    with report.open('x',encoding='utf8') as stream:stream.write(text)
    display(Markdown(text))
    print('Saved:',report)
else:
    print('Belum ada eksekusi. Laporan hasil/timing tidak dibuat-buat.')


**STOP — Tahap1b saja.** Tidak ada sel CV tuning, external evaluation, draft paper atau addendum.

Simpan notebook dengan output setelah Anda menjalankan. Semua artefak memakai prefix exploratory/development.
Jangan jalankan ulang sel initialization/fit/diagnostics/report ke output yang sama; no overwrite/resume berlaku.
Jika interruption/technical failure: status yang belum dijalankan tetap terlihat, artefak parsial dipertahankan.
Addendum yang mengunci delta/H2/domain dan persetujuan Tahap2 wajib sebelum pekerjaan berikutnya.